# 表格数据处理综合实践

学习目标：从原始订单、明细和事件完成读取、清洗、键校验、关联、汇总与导出，保留问题记录并核对结果。

前置知识：数据读写、dtype、缺失与重复、表格连接、分组重塑、时间与数据质量检查。

运行环境：Python 3.12、pandas 3.0、PyArrow 25.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

主例输入在单元内给出，date/ 中另保留同一批原始 CSV，后续单元沿用 pd 和已导入的标准库名称。文本显式使用 string，当前环境存储后端为 pyarrow；CSV 使用 C 引擎。临时往返文件放入 TemporaryDirectory，完成读写后清理，配套原始文件保留；不访问外部服务。

配套脚本：本章无外部脚本；数据位于 date/，为本章自制订单数据。

（1）[订单](date/22-orders.csv)、[明细](date/22-lines.csv)、[事件](date/22-events.csv)：可先检查原始文本，再对照读取结果；字段和异常口径见第 1 节。

## 1 输入与处理口径

本次交付包含订单摘要、客户分类金额表和问题记录。先明确这些约定，再处理数据。

（1）订单号、客户号、明细号、事件号是文本键。键去首尾空白并转大写；空字符串视为缺失。相同业务字段的重复记录保留首次；同键但内容冲突时停止自动处理。

（2）数量是 1～1,000 的整数，单价是 0～1,000,000 的整数分，明细金额为数量乘单价。输入文本去首尾空白后，只接受 ASCII 数字组成的规范整数写法：除 0 本身外没有前导零，不接受正负号、小数点和科学记数法。因此 12.0、+12、012、1e2 也不符合本任务的输入格式；这属于业务约定，不是 pandas 对数字的通用限制。不把缺失金额填为零，不把小数分四舍五入后接受。

（3）所有时间文本带 UTC 偏移，统一解析到 UTC。事件关联只允许同客户、严格早于下单、时间差不超过 5 分钟的记录，并假定事件在其时间戳所示时刻可获得。

（4）金额表只汇总通过校验且找到有效订单的明细，因此叫 accepted_cents。它不是存在错误明细时的完整订单金额；有被拒绝明细或没有有效明细的订单需要复核。

### 1.1 自制原始数据

输入故意包含重复记录、空键、非法日期、非法金额、缺失数量、小数分和不存在的订单号。原始文本保留，便于从问题记录回查。

In [1]:
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

orders_csv = '''order_id,customer_id,ordered_at
O1,C1,2026-01-01T10:00:00+00:00
 o2 ,C2,2026-01-01T10:05:00+00:00
O2,C2,2026-01-01T10:05:00+00:00
O3,C1,2026-01-01T10:10:00+00:00
O4,C3,bad-time
,C4,2026-01-01T10:12:00+00:00
O5,C3,2026-01-01T10:20:00+00:00
'''
lines_csv = '''line_id,order_id,category,quantity,unit_cents
L1,O1,book,2,500
L2,O1,food,1,300
L3,O2,book,1,700
L4,O3,food,3,200
L4,O3,food,3,200
L5,O9,book,1,900
L6,O2,food,1,bad
L7,,book,1,100
L8,O3,book,,400
L9,O3,book,1,12.5
'''
events_csv = '''event_id,customer_id,event_at,channel
E1,C1,2026-01-01T09:59:00+00:00,ad
E2,C2,2026-01-01T10:05:00+00:00,email
E3,C1,2026-01-01T10:11:00+00:00,search
E4,C3,2026-01-01T10:18:00+00:00,ad
'''

### 1.2 读取并保留来源

先按 string 读取，避免非法数字在进入问题记录前就中断读取。关闭默认缺失词表后，仅把空字段识别为缺失，避免将可能的文本编号误判为空。

每张表增加 source_row，表示本例 CSV 的原始行号。本例没有多行字段和空白行，因此数据索引加 2 对应文件行号。Path.write_text 完成后关闭文件；退出临时目录时，原始 CSV 已读入内存并被清理。

In [2]:
raw_tables = {}
with TemporaryDirectory() as directory:
    for name, text in [("orders", orders_csv), ("lines", lines_csv), ("events", events_csv)]:
        path = Path(directory) / f"{name}.csv"
        path.write_text(text, encoding="utf-8")
        frame = pd.read_csv(
            path, dtype="string", keep_default_na=False, na_values=[""],
            engine="c", encoding="utf-8",
        )
        frame["source_row"] = pd.Series(frame.index + 2, dtype="Int64")
        raw_tables[name] = frame
orders_raw = raw_tables["orders"]
lines_raw = raw_tables["lines"]
events_raw = raw_tables["events"]
print(orders_raw[["order_id", "customer_id", "source_row"]])
print({name: len(frame) for name, frame in raw_tables.items()})  # 7、10、4 行。
print(orders_raw["order_id"].dtype.storage)  # pyarrow。
print(Path(directory).exists())  # False：后续不依赖这些临时文件。

  order_id customer_id  source_row
0       O1          C1           2
1      o2           C2           3
2       O2          C2           4
3       O3          C1           5
4       O4          C3           6
5     <NA>          C4           7
6       O5          C3           8
{'orders': 7, 'lines': 10, 'events': 4}
pyarrow
False


### 1.3 对照配套原始文件

可以直接查看课程 date/ 中的三份 CSV，辨认重复记录、空键和非法金额。下面用相同参数读取持久文件，并与刚才的原始表逐项比较；后续清洗仍使用已经得到的 raw_tables。

In [3]:
for name in ["orders", "lines", "events"]:
    supplied = pd.read_csv(
        f"date/22-{name}.csv", dtype="string", keep_default_na=False,
        na_values=[""], engine="c", encoding="utf-8",
    )
    supplied["source_row"] = pd.Series(supplied.index + 2, dtype="Int64")
    pd.testing.assert_frame_equal(supplied, raw_tables[name], check_exact=True)
    print(name, len(supplied))  # orders 7、lines 10、events 4，原文与内联样例一致。

orders 7
lines 10
events 4


## 2 规范化订单并记录问题

### 2.1 键与时间

复制原表后清理键，原始值仍留在 orders_raw。日期解析失败转为 NaT，只用于识别问题，不代表可以继续把该行作为有效订单。

In [4]:
orders = orders_raw.copy()
for column in ["order_id", "customer_id"]:
    orders[column] = orders[column].str.strip().str.upper().replace("", pd.NA)
orders["ordered_at"] = pd.to_datetime(
    orders["ordered_at"], format="ISO8601", errors="coerce", utc=True,
)
print(orders[["order_id", "ordered_at"]])  # 两个 O2 已同形；O4 的时间为 NaT。
print(orders["ordered_at"].dtype)  # 本例为 datetime64[us, UTC]。

  order_id                ordered_at
0       O1 2026-01-01 10:00:00+00:00
1       O2 2026-01-01 10:05:00+00:00
2       O2 2026-01-01 10:05:00+00:00
3       O3 2026-01-01 10:10:00+00:00
4       O4                       NaT
5     <NA> 2026-01-01 10:12:00+00:00
6       O5 2026-01-01 10:20:00+00:00
datetime64[us, UTC]


### 2.2 问题优先级与重复处理

每条被拒绝记录先记一个主要原因：缺失键优先，其次非法日期，最后是完全重复。只有规范化后的全部业务字段相同才按重复记录保留首次，source_row 不参与重复判定。

筛选后检查订单号是否唯一；若同号仍有不同内容，就停止，不能任意选一条连接。本例第二条 O2 是完全重复，另有一条非法日期和一条空订单号。

In [5]:
order_reason = pd.Series("", index=orders.index, dtype="string")
order_reason.loc[orders[["order_id", "customer_id"]].isna().any(axis=1)] = "missing_key"
order_reason.loc[order_reason.eq("") & orders["ordered_at"].isna()] = "invalid_time"
order_fields = ["order_id", "customer_id", "ordered_at"]
duplicate_order = orders.duplicated(order_fields, keep="first")
order_reason.loc[order_reason.eq("") & duplicate_order] = "exact_duplicate"
bad_order = order_reason.ne("")
order_log = pd.DataFrame({
    "table": "orders", "source_row": orders.loc[bad_order, "source_row"],
    "key": orders.loc[bad_order, "order_id"], "reason": order_reason.loc[bad_order],
    "raw_value": orders_raw.loc[bad_order, "ordered_at"],
})
orders_clean = orders.loc[~bad_order, order_fields].reset_index(drop=True)
assert orders_clean["order_id"].is_unique, "同一订单号还有冲突记录，应停止并核查"
print(order_log[["source_row", "key", "reason"]])  # 原始第 4、6、7 行。
print(orders_clean["order_id"].tolist())  # O1、O2、O3、O5。

   source_row   key           reason
2           4    O2  exact_duplicate
4           6    O4     invalid_time
5           7  <NA>      missing_key
['O1', 'O2', 'O3', 'O5']


### 2.3 幂等与复现的区别

本流程约定“去首尾空白、转大写、空串转缺失”以及“完全重复保留首次”重复执行后不再改变数据，这称为幂等（idempotence）。只对这些约定步骤检查幂等，不能据此要求所有计算都幂等。

复现则是从同一份原始输入、相同规则和环境重新运行整个流程，得到相同输出；不是把最终结果再次当成原始订单输入。

In [6]:
normalized_again = orders["order_id"].str.strip().str.upper().replace("", pd.NA)
print(normalized_again.equals(orders["order_id"]))  # True。
deduplicated = orders.drop_duplicates(order_fields, keep="first")
print(deduplicated.drop_duplicates(order_fields, keep="first").equals(deduplicated))  # True。

True
True


## 3 清洗明细并计算整数金额

### 3.1 在原始文本上检查整数格式

明细键执行同样的文本规则。数量和单价先检查原始文本格式，再直接转成可空整数 Int64，最后检查各自范围；全过程不先转成浮点数。原始的 bad、空数量和 12.5 分都要留下问题记录。

str.fullmatch 要求整段文本符合正则表达式。下面模式中的 0 表示零，竖线表示二选一，[1-9] 表示首位非零，[0-9]{0,6} 表示后续最多六位数字。数量和单价的上限都不超过七位，因此先排除更长文本，再转换，避免超出 Int64 范围的文本使整批失败。where 将不符合条件的位置保留为缺失；na=False 把缺失文本判为格式不合格。

本章用一个小函数复用这两列的规则；它只服务于本任务的非负、最多七位整数范围，不是任意数值的通用解析器。

In [7]:
def parse_integer_field(text, lower, upper):
    """按本章最多七位的整数文本约定解析，非法或越界值返回缺失。"""
    stripped = text.str.strip()
    valid_format = stripped.str.fullmatch(r"0|[1-9][0-9]{0,6}", na=False)
    integers = stripped.where(valid_format).astype("Int64")
    return integers.where(integers.between(lower, upper))


lines = lines_raw.copy()
for column in ["line_id", "order_id"]:
    lines[column] = lines[column].str.strip().str.upper().replace("", pd.NA)
lines["category"] = lines["category"].str.strip().replace("", pd.NA)
quantity = parse_integer_field(lines["quantity"], 1, 1000)
unit_cents = parse_integer_field(lines["unit_cents"], 0, 1000000)
valid_quantity = quantity.notna()
valid_price = unit_cents.notna()
print(pd.DataFrame({"line_id": lines["line_id"], "quantity_ok": valid_quantity, "price_ok": valid_price}))
# L6、L9 单价不合格，L8 数量不合格；空外键 L7 将在键检查中排除。

  line_id  quantity_ok  price_ok
0      L1         True      True
1      L2         True      True
2      L3         True      True
3      L4         True      True
4      L4         True      True
5      L5         True      True
6      L6         True     False
7      L7         True      True
8      L8        False      True
9      L9         True     False


### 3.2 小数精度与范围边界

浮点表示的精度有限。先把文本转换为浮点，再用余数是否为零判断整数，检查的已经是舍入后的数值，不能据此证明原始文本满足整数条件。下面前三个小数字符串在当前环境经 to_numeric 转换后变成整数，属于应避免的校验方式；正式流程始终检查原文。

同时核对格式合法但越界、零单价、上下限、缺失及超长整数。拒绝记录保留原文，不能把解析产生的缺失值作为原始输入覆盖回去。

In [8]:
precision_text = pd.Series(
    ["12.0000000000000001", "0.99999999999999999", "1000000.00000000001"],
    dtype="string",
)
rounded_values = pd.to_numeric(precision_text, errors="coerce")
print(rounded_values.tolist())  # 当前环境为 [12.0, 1.0, 1000000.0]，原文小数被舍入。
print(rounded_values.mod(1).eq(0).tolist())  # [True, True, True]，不能作为原文合格的证据。
assert parse_integer_field(precision_text, 0, 1000000).isna().all()

boundary_text = pd.Series(
    ["0", "1", "1000", "1001", "1000000", "1000001", " 12 ", "12.0",
     "+12", "012", "1e2", "999999999999999999999999", "", pd.NA],
    dtype="string",
)
boundary_quantity = parse_integer_field(boundary_text, 1, 1000)
boundary_price = parse_integer_field(boundary_text, 0, 1000000)
print(pd.DataFrame({
    "raw_text": boundary_text, "quantity": boundary_quantity, "unit_cents": boundary_price,
}))
# 数量接受 1、1000、首尾带空白的 12；单价还接受 0、1001、1000000。
# 小数、符号、前导零、指数、超长文本、空文本和缺失均被拒绝。
assert boundary_quantity.notna().tolist() == [
    False, True, True, False, False, False, True, False, False, False, False, False, False, False,
]
assert boundary_price.notna().tolist() == [
    True, True, True, True, True, False, True, False, False, False, False, False, False, False,
]

[12.0, 1.0, 1000000.0]
[True, True, True]
                    raw_text  quantity  unit_cents
0                          0      <NA>           0
1                          1         1           1
2                       1000      1000        1000
3                       1001      <NA>        1001
4                    1000000      <NA>     1000000
5                    1000001      <NA>        <NA>
6                        12         12          12
7                       12.0      <NA>        <NA>
8                        +12      <NA>        <NA>
9                        012      <NA>        <NA>
10                       1e2      <NA>        <NA>
11  999999999999999999999999      <NA>        <NA>
12                                <NA>        <NA>
13                      <NA>      <NA>        <NA>


### 3.3 保留拒绝原因

明细也只删除业务字段完全相同的重复行，不直接按明细号随意去重。保留原始行号与“原数量 | 原单价”，便于解释为什么该行没有进入金额小计。

In [9]:
line_reason = pd.Series("", index=lines.index, dtype="string")
missing_line_key = lines[["line_id", "order_id", "category"]].isna().any(axis=1)
line_reason.loc[missing_line_key] = "missing_key"
line_reason.loc[line_reason.eq("") & ~(valid_quantity & valid_price)] = "invalid_number"
line_fields = ["line_id", "order_id", "category", "quantity", "unit_cents"]
duplicate_line = lines.duplicated(line_fields, keep="first")
line_reason.loc[line_reason.eq("") & duplicate_line] = "exact_duplicate"
bad_line = line_reason.ne("")
raw_numbers = lines_raw["quantity"].fillna("<empty>") + " | " + lines_raw["unit_cents"].fillna("<empty>")
line_log = pd.DataFrame({
    "table": "lines", "source_row": lines.loc[bad_line, "source_row"],
    "key": lines.loc[bad_line, "line_id"], "reason": line_reason.loc[bad_line],
    "raw_value": raw_numbers.loc[bad_line],
})
lines_clean = lines.loc[~bad_line].copy()
lines_clean["quantity"] = quantity.loc[~bad_line].astype("Int64")
lines_clean["unit_cents"] = unit_cents.loc[~bad_line].astype("Int64")
assert lines_clean["line_id"].is_unique, "同一明细号还有冲突记录，应停止并核查"
lines_clean["line_cents"] = lines_clean["quantity"] * lines_clean["unit_cents"]
print(line_log[["source_row", "key", "reason", "raw_value"]])  # 5 条问题。
print(lines_clean[["line_id", "order_id", "line_cents"]])  # L1～L5；L5 尚未验证订单关系。
print(lines_clean["line_cents"].dtype)  # Int64，金额仍是整数分。

   source_row key           reason      raw_value
4           6  L4  exact_duplicate        3 | 200
6           8  L6   invalid_number        1 | bad
7           9  L7      missing_key        1 | 100
8          10  L8   invalid_number  <empty> | 400
9          11  L9   invalid_number       1 | 12.5
  line_id order_id  line_cents
0      L1       O1        1000
1      L2       O1         300
2      L3       O2         700
3      L4       O3         600
5      L5       O9         900
Int64


## 4 连接并核对键关系

### 4.1 明细对应唯一订单

一张订单可以有多条明细，每条明细只能对应一张有效订单，因此连接使用 validate="m:1"。indicator 区分是否匹配，不根据金额是否缺失推断关系。

两侧键已排除缺失，避免 pandas 将两侧空键相互匹配。找不到有效订单的明细单独记录，不进入客户金额汇总。

In [10]:
joined = lines_clean.merge(
    orders_clean, on="order_id", how="left", validate="m:1", indicator=True,
)
orphan = joined["_merge"].eq("left_only")
orphan_log = pd.DataFrame({
    "table": "lines", "source_row": joined.loc[orphan, "source_row"],
    "key": joined.loc[orphan, "line_id"], "reason": "unmatched_order",
    "raw_value": joined.loc[orphan, "order_id"],
})
accepted = joined.loc[~orphan].drop(columns="_merge").reset_index(drop=True)
assert len(joined) == len(lines_clean) == 5
assert accepted["line_cents"].sum() == 2600
print(joined[["line_id", "order_id", "_merge"]])  # L5 的 O9 没有有效订单。
print(accepted[["line_id", "customer_id", "line_cents"]])  # 1000、300、700、600 分。
print(len(accepted), len(orphan_log))  # 4 条接受，1 条关系问题。

  line_id order_id     _merge
0      L1       O1       both
1      L2       O1       both
2      L3       O2       both
3      L4       O3       both
4      L5       O9  left_only
  line_id customer_id  line_cents
0      L1          C1        1000
1      L2          C1         300
2      L3          C2         700
3      L4          C1         600
4 1


### 4.2 重复主键必须阻止连接

下面故意给订单表再加一条 O1，验证关系检查确实会失败；不把连接后的行数膨胀当成新增销售。

In [11]:
duplicate_parent = pd.concat([orders_clean, orders_clean.iloc[[0]]], ignore_index=True)
try:
    lines_clean.merge(duplicate_parent, on="order_id", how="left", validate="m:1")
except pd.errors.MergeError as error:
    print(type(error).__name__, str(error))  # 右侧键不唯一，不满足 many-to-one。
else:
    raise AssertionError("重复订单主键应使连接失败")

MergeError Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 order_id
      O1 ...


## 5 汇总与重塑

### 5.1 保留没有有效明细的订单

按订单汇总接受的明细，再左连接回有效订单。accepted_lines 为零表示没有接受的明细；accepted_cents 保留缺失，不宣称该订单真实金额为零。

O2、O3 有非法明细，O5 没有有效明细，所以需要复核。完全重复记录的副本已按规则处理，本身不作为该标记的依据。

In [12]:
order_totals = accepted.groupby(
    "order_id", as_index=False, sort=True, dropna=False, observed=True,
).agg(accepted_cents=("line_cents", "sum"), accepted_lines=("line_id", "size"))
order_report = orders_clean.merge(order_totals, on="order_id", how="left", validate="1:1")
order_report["accepted_lines"] = order_report["accepted_lines"].fillna(0).astype("Int64")
rejected_order_ids = lines.loc[bad_line & line_reason.ne("exact_duplicate"), "order_id"].dropna()
order_report["needs_review"] = (
    order_report["order_id"].isin(rejected_order_ids) | order_report["accepted_lines"].eq(0)
).astype("boolean")
print(order_report[["order_id", "accepted_cents", "accepted_lines", "needs_review"]])
# O1：1300、2、False；O2：700、1、True；O3：600、1、True；O5：<NA>、0、True。
assert order_report["accepted_cents"].sum() == accepted["line_cents"].sum()
print(order_report.shape)  # (4, 6)，一张有效订单一行。

  order_id  accepted_cents  accepted_lines  needs_review
0       O1            1300               2         False
1       O2             700               1          True
2       O3             600               1          True
3       O5            <NA>               0          True


(4, 6)


### 5.2 客户与分类金额表

先按客户、分类汇总，确保每个组合只有一行，再用 pivot 转成宽表。本表只描述接受的明细；某个已出现客户没有某类接受明细时填 0，不据此判断其真实订单是否完整。

In [13]:
customer_totals = accepted.groupby(
    ["customer_id", "category"], as_index=False, sort=True, dropna=False, observed=True,
).agg(accepted_cents=("line_cents", "sum"))
customer_matrix = customer_totals.pivot(
    index="customer_id", columns="category", values="accepted_cents",
).fillna(0).astype("Int64")
print(customer_totals)  # C1：book 1000、food 900；C2：book 700。
print(customer_matrix)  # 行 C1、C2；列 book、food；C2 的 food 为 0。
assert customer_matrix.sum().sum() == 2600
print(customer_matrix.shape)  # (2, 2)；无接受明细的 C3 不在该表。

  customer_id category  accepted_cents
0          C1     book            1000
1          C1     food             900
2          C2     book             700


category     book  food
customer_id            
C1           1000   900
C2            700     0
(2, 2)


## 6 时间关联与未来信息

### 6.1 准备事件键

本例事件输入全部有效，先检查必需字段、事件号唯一性和同客户同时间的重复，再执行匹配。若检查失败应先形成事件问题记录，不能直接把空时间交给 merge_asof。

按时间键全局排序，两侧都统一到 UTC；指定 by 也不能只保证客户组内有序。

In [14]:
events = events_raw.copy()
for column in ["event_id", "customer_id"]:
    events[column] = events[column].str.strip().str.upper().replace("", pd.NA)
events["event_at"] = pd.to_datetime(events["event_at"], format="ISO8601", errors="coerce", utc=True)
required_events = ["event_id", "customer_id", "event_at", "channel"]
assert not events[required_events].isna().any().any()
assert events["event_id"].is_unique
assert not events.duplicated(["customer_id", "event_at"]).any()
events = events[required_events].sort_values("event_at").reset_index(drop=True)
timed_orders = order_report.sort_values("ordered_at").reset_index(drop=True)
print(events[["event_id", "customer_id", "event_at"]])
print(timed_orders["ordered_at"].dtype, events["event_at"].dtype)
# 两侧均为 datetime64[us, UTC]，时间键非空且全局递增。

  event_id customer_id                  event_at
0       E1          C1 2026-01-01 09:59:00+00:00
1       E2          C2 2026-01-01 10:05:00+00:00
2       E3          C1 2026-01-01 10:11:00+00:00
3       E4          C3 2026-01-01 10:18:00+00:00
datetime64[us, UTC] datetime64[us, UTC]


### 6.2 严格使用下单前事件

backward 只向历史寻找候选，allow_exact_matches=False 排除同时刻事件，tolerance 限制为 5 分钟。保留两侧时间，核对每个已匹配时间差。

未匹配不是删除订单的理由：O2 的事件与下单同时发生，O3 的历史事件又太早，均保留空事件字段。

In [15]:
report = pd.merge_asof(
    timed_orders, events, by="customer_id", left_on="ordered_at", right_on="event_at",
    direction="backward", tolerance=pd.Timedelta("5min"), allow_exact_matches=False,
)
matched_time = report["event_at"].notna()
age = report.loc[matched_time, "ordered_at"] - report.loc[matched_time, "event_at"]
assert age.gt(pd.Timedelta(0)).all() and age.le(pd.Timedelta("5min")).all()
assert len(report) == len(order_report)
print(report[["order_id", "event_id", "channel"]])  # O1→E1，O5→E4；O2、O3 未匹配。
print(age.tolist())  # 匹配事件分别早 1 分钟、2 分钟。
print(report.loc[~matched_time, "order_id"].tolist())  # ['O2', 'O3']。

  order_id event_id channel
0       O1       E1      ad
1       O2     <NA>    <NA>
2       O3     <NA>    <NA>
3       O5       E4      ad


[Timedelta('0 days 00:01:00'), Timedelta('0 days 00:02:00')]
['O2', 'O3']


### 6.3 最近不等于历史最近

如果改成 nearest，10:10 的 O3 会找到 10:11 的 E3。它虽然只相差 1 分钟，却发生在下单之后；对当前“下单前可用信息”的任务属于未来信息，不能采用。

下面只做反例观察，不替换最终 report。

In [16]:
future_example = pd.merge_asof(
    timed_orders.loc[timed_orders["order_id"].eq("O3")], events,
    by="customer_id", left_on="ordered_at", right_on="event_at",
    direction="nearest", tolerance=pd.Timedelta("5min"), allow_exact_matches=False,
)
print(future_example[["order_id", "ordered_at", "event_id", "event_at"]])  # O3→E3。
print((future_example["event_at"] > future_example["ordered_at"]).tolist())  # [True]。

  order_id                ordered_at event_id                  event_at
0       O3 2026-01-01 10:10:00+00:00       E3 2026-01-01 10:11:00+00:00


[True]


## 7 问题记录与空输入

### 7.1 汇总可回查的问题记录

合并订单、明细及未匹配订单号的问题。table 与 source_row 指向原始输入，key 是规范化后的记录键，raw_value 保留时间文本、数量与单价文本或未匹配订单号。规范化和问题判断不会覆盖原始表。

In [17]:
issues = pd.concat([order_log, line_log, orphan_log], ignore_index=True)
issues = issues.astype({
    "table": "string", "source_row": "Int64", "key": "string",
    "reason": "string", "raw_value": "string",
})
issues = issues.sort_values(["table", "source_row"]).reset_index(drop=True)
print(issues)  # 订单 3 条、明细清洗 5 条、关系问题 1 条，共 9 条。
assert len(orders_raw) == len(orders_clean) + len(order_log)
assert len(lines_raw) == len(accepted) + len(line_log) + len(orphan_log)
print(issues.groupby("table", sort=True, dropna=False).size())  # lines 6，orders 3。

    table  source_row   key           reason                  raw_value
0   lines           6    L4  exact_duplicate                    3 | 200
1   lines           7    L5  unmatched_order                         O9
2   lines           8    L6   invalid_number                    1 | bad
3   lines           9    L7      missing_key                    1 | 100
4   lines          10    L8   invalid_number              <empty> | 400
5   lines          11    L9   invalid_number                   1 | 12.5
6  orders           4    O2  exact_duplicate  2026-01-01T10:05:00+00:00
7  orders           6    O4     invalid_time                   bad-time
8  orders           7  <NA>      missing_key  2026-01-01T10:12:00+00:00
table
lines     6
orders    3
dtype: int64


### 7.2 零行与没有有效金额

只有表头的 CSV 是零行表，不等于含缺失值的非空表。空批次仍要保留列定义，不对第一行作无条件访问。

下面先读取只有表头的明细，再用已确定类型的零行明细检查汇总边界：没有可接受明细时，订单仍保留，金额小计缺失。直接对空数列求和可能得到 0，因此不能把这个默认值当成“已知零金额”。

In [18]:
with StringIO(lines_csv.splitlines()[0] + "\n") as buffer:
    empty_input = pd.read_csv(buffer, dtype="string", engine="c")
print(empty_input.shape, empty_input.empty)  # (0, 5)，True。
empty_accepted = accepted.iloc[:0].copy()
empty_totals = empty_accepted.groupby(
    "order_id", as_index=False, sort=True, dropna=False, observed=True,
).agg(accepted_cents=("line_cents", "sum"))
empty_report = orders_clean.merge(empty_totals, on="order_id", how="left", validate="1:1")
print(empty_report[["order_id", "accepted_cents"]])  # 四张订单都保留，金额全为 <NA>。
print(empty_accepted["line_cents"].sum(), empty_accepted["line_cents"].sum(min_count=1))
# 默认和为 0，要求至少一个有效值时为 <NA>；本章使用后者的缺失含义。

(0, 5) True
  order_id  accepted_cents
0       O1            <NA>
1       O2            <NA>
2       O3            <NA>
3       O5            <NA>
0 <NA>


## 8 抽查与导出往返

### 8.1 固定输入顺序的抽查

全部键、金额与关联检查都在全量数据上完成；抽样只用于查看个别明细。先按唯一明细号排序，再固定 random_state，重复抽样才基于同一输入顺序。固定种子不意味着改变输入或环境后永远得到同一结果。

样本量取 2 与有效行数的较小值，不放回，兼顾小批次与零行批次。

In [19]:
review_input = accepted.sort_values("line_id").reset_index(drop=True)
sample_size = min(2, len(review_input))
sample = review_input.sample(n=sample_size, replace=False, random_state=7)
sample_again = review_input.sample(n=sample_size, replace=False, random_state=7)
pd.testing.assert_frame_equal(sample, sample_again)
print(sample[["line_id", "order_id", "line_cents"]])  # 同一固定输入下两次结果一致。
print(sample["line_id"].is_unique, len(sample))  # True，2。

  line_id order_id  line_cents
2      L3       O2         700
1      L2       O1         300
True 2


### 8.2 明确输出列与类型

最终订单表保留金额口径、复核标记和两侧时间。按订单号固定交付顺序；客户分类表把行索引还原为普通列，去掉列轴名称，并把列标签统一为 str 类型，便于 CSV 往返。列标签的类型与各数据列的类型是两回事。

In [20]:
report_columns = [
    "order_id", "customer_id", "ordered_at", "accepted_cents", "accepted_lines",
    "needs_review", "event_id", "event_at", "channel",
]
report = report[report_columns].sort_values("order_id").reset_index(drop=True)
matrix_export = customer_matrix.rename_axis(columns=None).reset_index()
matrix_export.columns = pd.Index(matrix_export.columns, dtype="str")
print(report[["order_id", "accepted_cents", "needs_review", "event_id"]])
print(report.dtypes)  # string、UTC 时间、Int64、boolean；不把缺失金额变成 0。
print(matrix_export)  # customer_id 为普通列，book、food 为 Int64。

  order_id  accepted_cents  needs_review event_id
0       O1            1300         False       E1
1       O2             700          True     <NA>
2       O3             600          True     <NA>
3       O5            <NA>          True       E4
order_id                       string
customer_id                    string
ordered_at        datetime64[us, UTC]
accepted_cents                  Int64
accepted_lines                  Int64
needs_review                  boolean
event_id                       string
event_at          datetime64[us, UTC]
channel                        string
dtype: object
  customer_id  book  food
0          C1  1000   900
1          C2   700     0


### 8.3 导出结果与问题记录

CSV 不保存 pandas 类型元数据。读回时明确恢复文本、可空整数、布尔与 UTC 时间，再用 assert_frame_equal 同时检查值、标签、顺序和 dtype；仅比较金额总和不足以验证往返。

本例时间精确到整秒，按带偏移的格式导出。若原数据含更细精度，必须相应调整格式并重新检查。文件在临时目录内完成读回；上下文结束后全部清理。

In [21]:
with TemporaryDirectory() as directory:
    report_path = Path(directory) / "orders_report.csv"
    matrix_path = Path(directory) / "customer_matrix.csv"
    issues_path = Path(directory) / "issues.csv"
    time_format = "%Y-%m-%dT%H:%M:%S%z"
    report.to_csv(report_path, index=False, encoding="utf-8", date_format=time_format)
    matrix_export.to_csv(matrix_path, index=False, encoding="utf-8")
    issues.to_csv(issues_path, index=False, encoding="utf-8")
    report_types = {
        "order_id": "string", "customer_id": "string", "ordered_at": "string",
        "accepted_cents": "Int64", "accepted_lines": "Int64", "needs_review": "boolean",
        "event_id": "string", "event_at": "string", "channel": "string",
    }
    restored = pd.read_csv(report_path, dtype=report_types, engine="c", encoding="utf-8")
    for column in ["ordered_at", "event_at"]:
        restored[column] = pd.to_datetime(restored[column], format=time_format, utc=True)
    restored_matrix = pd.read_csv(
        matrix_path, dtype={"customer_id": "string", "book": "Int64", "food": "Int64"},
        engine="c", encoding="utf-8",
    )
    restored_issues = pd.read_csv(
        issues_path, dtype=issues.dtypes.astype(str).to_dict(), engine="c", encoding="utf-8",
    )
    pd.testing.assert_frame_equal(restored, report)
    pd.testing.assert_frame_equal(restored_matrix, matrix_export)
    pd.testing.assert_frame_equal(restored_issues, issues)
    print(restored.shape, restored_matrix.shape, restored_issues.shape)  # (4, 9)、(2, 3)、(9, 5)。
    print("三张表的值、标签、顺序和类型往返一致")
print(Path(directory).exists())  # False：输出文件与目录都已清理。

(4, 9) (2, 3) (9, 5)
三张表的值、标签、顺序和类型往返一致
False


## 本章小结

（1）先确定键、整数文本格式、金额单位、数值范围、时间时区和拒绝规则；整数校验先检查原文，保留原始输入。

（2）清洗问题、连接未匹配和没有有效明细是不同情况；问题记录应能回到原始行，部分金额不能冒充完整金额。

（3）连接前检查唯一性与缺失键，连接时验证基数，汇总后核对手算金额及各阶段行数。

（4）时间匹配明确分组、方向、容差和相等边界；保留两侧时间可发现未来信息。

（5）幂等检查针对约定步骤，复现从同一原始输入开始；导出往返同时检查值、标签、顺序和类型。

## 练习

（1）对下面两条明细应用数量、单价范围与整数分规则，区分接受与拒绝记录。记录问题原因，不先四舍五入；计算接受明细的小计。

In [22]:
exercise_lines = pd.DataFrame({
    "line_id": ["X1", "X2"], "quantity": ["2", "1"], "unit_cents": ["250", "12.5"],
}, dtype="string")
# 在此解析、检查并计算；保留原文本和拒绝原因。
# 检查：只接受 X1，小计 500 分；X2 的小数分不得静默改为整数。

（2）先预测下面连接的行数与金额和，再运行核对。若右表应该每张订单一行，应增加什么约束？写出能阻止错误汇总的连接，并具体捕获预期异常。

In [23]:
exercise_detail = pd.DataFrame({"order_id": ["A"], "line_cents": [500]})
exercise_parent = pd.DataFrame({"order_id": ["A", "A"], "customer_id": ["C1", "C1"]})
prediction = exercise_detail.merge(exercise_parent, on="order_id", how="left")
print(len(prediction), prediction["line_cents"].sum())
# 先写预测，再解释金额变化是否来自新增交易。
# 在此添加 validate，并用具体异常类型与 else 检查预期失败。

2 1000


（3）沿用本章 timed_orders 和 events。任务改成“事件发生时间可以等于下单时间，但仍不得晚于下单，容差仍为 5 分钟”。修改参数并解释选择理由，指出哪张订单新增匹配，为什么 O3 仍不能采用 E3。

In [24]:
# 在此从相同的 timed_orders 和 events 重新计算，不修改最终 report。
# 检查：O2 新增 E2；O1、O5 的原匹配不变；O3 仍未匹配。
# 保留两侧时间，检查时间差非负且不超过 5 分钟。

（4）条件进一步改成“有任何非法明细的订单整单不进入金额汇总”。完全重复副本不算非法明细，但没有有效明细的订单仍需单独报告。根据本章接受明细与拒绝记录重新计算客户金额，并说明这种口径与 accepted_cents 的区别。

In [25]:
# 可使用 accepted、lines、line_reason 和 orders_clean。
# 在此识别含非法明细的订单，筛选接受明细，再重新分组。
# 检查：O2、O3 被整单排除，O1 保留 1300 分，O5 无有效明细需单列。
# C1 的汇总仅剩 book 1000、food 300；解释不能直接沿用原客户分类表的原因。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档或 API 源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | 读写与类型：[read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) 的 dtype、keep_default_na、na_values、engine；[to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) 的 index、encoding、date_format；[to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) 的返回类型与精度警告（用于反例）；[to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) 的 format、errors、utc；[String migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#for-existing-users-of-the-nullable-stringdtype) 的显式 string 与缺失语义。清洗与边界：[str.fullmatch](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html) 的整段匹配与 na；[where](https://pandas.pydata.org/docs/reference/api/pandas.Series.where.html) 的条件保留和缺失替换；[astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) 的指定类型转换；[str.strip](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html)、[str.upper](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.upper.html) 的字符串规范化；[replace](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) 的按值替换；[between](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html) 的区间判断；[duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)、[drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) 的 subset、keep；[empty](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.empty.html) 的零行与全缺失区别；[sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) 的 min_count。关联与汇总：[merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) 的空键匹配、validate、indicator；[GroupBy — Named aggregation](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation) 的命名聚合；[groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) 的 as_index、sort、dropna、observed；[pivot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html) 的键组合唯一条件；[fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) 与 [isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) 的填充和成员判断；[merge_asof](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html) 的全局排序、by、direction、tolerance、allow_exact_matches。检查：[Index](https://pandas.pydata.org/docs/reference/api/pandas.Index.html) 的 dtype 参数与列标签类型；[sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) 的 n、replace、random_state；[assert_frame_equal](https://pandas.pydata.org/docs/reference/api/pandas.testing.assert_frame_equal.html) 的值、标签、dtype 等比较。整数文本格式、清洗优先级、金额范围及复核规则是本章自制任务的业务约定。 |
| Python 官方文档（3.12） | [正则表达式语法](https://docs.python.org/3.12/library/re.html#regular-expression-syntax) 的字符范围、分支与重复次数；[浮点数的表示误差](https://docs.python.org/3.12/tutorial/floatingpoint.html#representation-error) 的有限精度与舍入；[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的上下文清理；[Path.write_text](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.write_text) 的写入并关闭；[StringIO](https://docs.python.org/3.12/library/io.html#io.StringIO) 的内存文本流与关闭。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[groupby](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/groupby.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |